# **Etapa 02 - Modelagem com Redes Neurais**

**Objetivo:** Construir MLP utilizando PyTorch e comparar métricas do mesmo com modelos lineares e de árvores.

## **Imports e Setup Inicial de Valores**

In [ ]:
#!pip install mlflow

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import logging
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from IPython.display import display


RANDOM_STATE = 17
TEST_SIZE = 0.2

# Declaração do logger no escopo global
logger = logging.getLogger("tc_etapa_02")


sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## **Método para configuração do Logger**

In [ ]:
def configurar_logging(nivel=logging.INFO):
    """
    Configura o logger global. Pode ser chamado múltiplas vezes
    para resetar as configurações durante a sessão.
    """
    # Limpa handlers existentes para evitar duplicação de logs
    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(nivel)
    logger.propagate = False

    # Define o formato
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

    # Handler para o console (saída no notebook)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


## **Carregamento de Dados**

In [ ]:
def carregar_dados(path):
    """
    Carrega um arquivo Excel e retorna um DataFrame.
    """
    logger.info(f"Carregando dados do arquivo: {path}")
    df = pd.read_excel(path)
    return df

## **Tratamento dos dados**

In [ ]:
def padronizar_nomes_features(df):
    """
    Padroniza os nomes das colunas para lowercase e substitui espaços por underscores.
    """
    logger.info("Padronizando nomes das colunas")
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    return df

In [ ]:
def corrigir_feature_total_charges(df):
  """
  Corrige a feature 'total_charges' para float64 e substitui NaN por 0.
  """
  logger.info("Corrigindo feature 'total_charges'")
  df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
  df.fillna({'total_charges': 0}, inplace=True)
  return df

In [ ]:
def remover_features_irrelevantes(df):
  """
  Remove features irrelevantes para o modelo.
  """
  logger.info("Removendo features irrelevantes")

  cols_to_drop = [
    "customerid",
    "count",
    "country",
    "state",
    "city",
    "lat_long",
    "latitude",
    "longitude",
    "churn_label",
    "churn_score",
    "cltv",
    "churn_reason",
    #"total_charges",
    #"tenure_months"
  ]
  logger.debug(f"Features a serem removidas: {cols_to_drop}")
  df = df.drop(columns=cols_to_drop)
  return df

In [ ]:
def criar_feature_average_monthly_spend(df):
  """
  Cria a feature 'average_monthly_spend' a partir de 'total_charges' e 'tenure_months'.
  """
  logger.info("Criando feature 'average_monthly_spend'")
  df['average_monthly_spend'] = df['total_charges'] / df['tenure_months']
  df['average_monthly_spend'] = df['average_monthly_spend'].replace([np.inf, -np.inf], 0).fillna(0)
  return df

In [ ]:
def aplicar_feature_engineering(df):
  """
  Aplica todas as transformações de feature engineering.
  """
  logger.info("Aplicando feature engineering")
  df = criar_feature_average_monthly_spend(df)
  return df

In [ ]:
def tratar_dados(df):
    """
    Aplica todos os tratamentos de dados.
    """
    logger.info("Tratando dados")
    df = padronizar_nomes_features(df)
    df = corrigir_feature_total_charges(df)
    # df = aplicar_feature_engineering(df)
    df = remover_features_irrelevantes(df)
    return df

## **Separação de Dados de Treino e Teste**

In [ ]:
def separar_dados_treino_teste(df):
  """
  Separa os dados em treino e teste.
  """
  logger.info("Separando dados em treino e teste")

  X = df.drop('churn_value', axis=1)
  y = df['churn_value']

  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
  )

  return X_train, X_test, y_train, y_test

## **Construir Transformer para One-Hot Encoding**

In [ ]:
def construir_transformer_hot_encoding(X_train):
  """
  Cria um transformer para one-hot encoding.
  """
  logger.info("Construindo transformer para one-hot encoding")

  colunas_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
  colunas_categoricas = X_train.select_dtypes(include=['object']).columns

  transformer = ColumnTransformer(
      transformers=[
          ('numerics', StandardScaler(), colunas_numericas),
          ('categoricals', OneHotEncoder(drop='first', handle_unknown='ignore'), colunas_categoricas)
      ]
  )

  return transformer

## **Construir Balanceador**

In [ ]:
def construir_balanceador():
  """
  Cria um balanceador SMOTE.
  """
  logger.info("Construindo balanceador SMOTE")

  return SMOTE(random_state=RANDOM_STATE)

## **Construir Pipeline**

In [ ]:
def construir_pipeline(nome_execucao, transformer, balanceador, modelo):
  """
  Cria um pipeline com o transformer, balanceador e modelo.
  """
  logger.info(f"Construindo pipeline para '{nome_execucao}'")

  pipeline = Pipeline(
    steps=[
        ('pre-processamento', transformer),
        ('balanceamento', balanceador),
        ('classificador', modelo)
    ]
  )

  return pipeline

## **Executar validação cruzada**

In [ ]:
def executar_validacao_cruzada(nome_execucao, pipeline, X_train, y_train, folds=5):
  """
  Executa a validação cruzada com o pipeline e retorna os resultados.
  """
  logger.info(f"Executando validação cruzada para '{nome_execucao}'")

  metricas = ['accuracy', 'precision', 'recall', 'f1']
  resultados_cv = cross_validate(
      pipeline, X_train, y_train, cv=5, scoring=metricas
  )

  logger.debug(f"--- FASE DE VALIDAÇÃO CRUZADA (Média das {folds} Pastas) ---")
  logger.debug(f"Acurácia Média:   {resultados_cv['test_accuracy'].mean():.4f}")
  logger.debug(f"Precisão Média: {resultados_cv['test_precision'].mean():.4f}")
  logger.debug(f"Recall Médio:    {resultados_cv['test_recall'].mean():.4f}")
  logger.debug(f"F1-Score Médio:  {resultados_cv['test_f1'].mean():.4f}\n")

## **Treinamento e Teste do Modelo**

In [ ]:
def treinar_e_testar_modelo(nome_execucao, pipeline, X_train, X_test, y_train, y_test):
  """
  Treina e testa o modelo.
  """
  logger.info(f"Treinando e testando modelo '{nome_execucao}'")

  pipeline.fit(X_train, y_train)

  previsoes = {}

  previsoes_train = pipeline.predict(X_train)
  previsoes_test = pipeline.predict(X_test)

  previsoes["previsoes_train"] = previsoes_train
  previsoes["previsoes_test"] = previsoes_test

  if (hasattr(pipeline, "predict_proba")):
    previsoes_train_proba = pipeline.predict_proba(X_train)[:, 1]
    previsoes_test_proba = pipeline.predict_proba(X_test)[:, 1]
  elif (hasattr(pipeline, "decision_function")):
    previsoes_train_proba = pipeline.decision_function(X_train)
    previsoes_test_proba = pipeline.decision_function(X_test)

  previsoes["previsoes_train_proba"] = previsoes_train_proba
  previsoes["previsoes_test_proba"] = previsoes_test_proba

  return previsoes

## **Calcular Métricas**

In [ ]:
def calcular_metricas(y_target, previsoes, label):
  """
  Calcula as métricas de avaliação.
  """
  logger.info("Calculando métricas")

  acuracia = accuracy_score(y_target, previsoes)
  precisao = precision_score(y_target, previsoes, pos_label=1)
  recall = recall_score(y_target, previsoes, pos_label=1)
  f1 = f1_score(y_target, previsoes, pos_label=1)

  logger.debug("--- MÉTRICAS DE EXECUÇÃO DO MODELO ---")
  logger.debug(f"Acurácia no {label}:  {acuracia:.4f}")
  logger.debug(f"Precisão no {label}: {precisao:.4f}")
  logger.debug(f"Recall no {label}:    {recall:.4f}")
  logger.debug(f"F1-Score no {label}:  {f1:.4f}\n")

  logger.debug("--- RELATÓRIO DE CLASSIFICAÇÃO DETALHADO ---")
  report = classification_report(y_target, previsoes)
  logger.debug(f"\n{report}")

  return acuracia, precisao, recall, f1



## **Calcular Métricas de Treino e Teste**

In [ ]:
def calcular_metricas_treino_teste(nome_execucao, y_train, y_test, previsoes):
  """
  Calcula as métricas de treino e teste.
  """
  logger.info(f"Calculando métricas de treino e teste para '{nome_execucao}")

  train_accuracy, train_precision, train_recall, train_f1 = calcular_metricas(y_train, previsoes["previsoes_train"], "TREINO")
  test_accuracy, test_precision, test_recall, test_f1 = calcular_metricas(y_test, previsoes["previsoes_test"], "TESTE")

  overfitting = train_accuracy - test_accuracy

  logger.debug("--- OVERFITTING ---")
  logger.debug(f"Overfitting: {overfitting:.4f}")

  metricas = {}
  metricas["train_accuracy"] = train_accuracy
  metricas["test_accuracy"] = test_accuracy
  metricas["train_precision"] = train_precision
  metricas["test_precision"] = test_precision
  metricas["train_recall"] = train_recall
  metricas["test_recall"] = test_recall
  metricas["train_f1"] = train_f1
  metricas["test_f1"] = test_f1
  metricas["overfitting"] = overfitting

  return metricas

## **Registrar Execução no MLFlow**

In [ ]:
def registrar_execucao_mlflow(
    nome_execucao,
    metricas,
    model,
    mlp = False
):
  """
  Registra a execução no MLFlow.
  """
  logger.info("Registrando execução no MLFlow")
  with mlflow.start_run(run_name=nome_execucao):
    mlflow.log_metric("train_accuracy", metricas["train_accuracy"])
    mlflow.log_metric("test_accuracy", metricas["test_accuracy"])
    mlflow.log_metric("train_precision", metricas["train_precision"])
    mlflow.log_metric("test_precision", metricas["test_precision"])
    mlflow.log_metric("train_recall", metricas["train_recall"])
    mlflow.log_metric("test_recall", metricas["test_recall"])
    mlflow.log_metric("train_f1_score", metricas["train_f1"])
    mlflow.log_metric("test_f1_score", metricas["test_f1"])
    mlflow.log_metric("overfitting", metricas["overfitting"])
    if mlp:
      mlflow.pytorch.log_model(model, "model")
    else:
      mlflow.sklearn.log_model(model, "model")

## **Registrar Métricas do Modelo**

In [ ]:
def registrar_metricas_modelo(modelo, metricas, metricas_todos_modelos):
  """
  Registra as métricas do modelo.
  """
  logger.info("Registrando métricas do modelo")
  metricas_todos_modelos.append({
      "Modelo": modelo,
      "Accuracy": metricas["test_accuracy"],
      "Precision": metricas["test_precision"],
      "Recall": metricas["test_recall"],
      "F1-Score": metricas["test_f1"],
      "Overfitting": metricas["overfitting"]
  })

## **Avaliar Modelo**

In [ ]:
def avaliar_modelo(nome_execucao, transformer, balanceador, modelo, X_train, y_train, X_test, y_test):
  """
  Avalia o modelo.
  """
  logger.info("Avaliando modelo")

  pipeline = construir_pipeline(nome_execucao, transformer, balanceador, modelo)

  executar_validacao_cruzada(nome_execucao, pipeline, X_train, y_train)

  previsoes = treinar_e_testar_modelo(nome_execucao, pipeline, X_train, X_test, y_train, y_test)

  metricas = calcular_metricas_treino_teste(nome_execucao, y_train, y_test, previsoes)

  registrar_execucao_mlflow(nome_execucao, metricas, modelo)

  return_data = {
    "modelo": nome_execucao,
    "metricas": metricas
  }

  return return_data

## **Configurar MLFlow**

In [ ]:
def configurar_mlflow():
  """
  Configura o MLFlow.
  """
  logger.info("Configurando MLFlow")
  os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
  mlflow.set_tracking_uri("/content/mlflow")
  mlflow.set_experiment("TechChallenge - Etapa 02")


## **Executar Modelo LogisticRegression**

In [ ]:
def executar_logistic_regression(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo LogisticRegression.
  """
  logger.info("Executando modelo LogisticRegression")

    # Obter modelo
  modelo = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight="balanced")

  # Avaliar modelo
  metricas = avaliar_modelo("Logistic Regression", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

  return metricas

## **Executar Modelo DecisionTreeClassifier**

In [ ]:
def executar_decision_tree_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo DecisionTreeClassifier.
  """
  logger.info("Executando modelo DecisionTreeClassifier")

  # Obter modelo
  modelo = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced", max_depth=5)

  # Avaliar modelo
  metricas = avaliar_modelo("Decision Tree Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

  return metricas

## **Executar Modelo RandomForestClassifier**

In [ ]:
def executar_random_forest_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo RandomForestClassifier.
  """
  logger.info("Executando modelo RandomForestClassifier")

  # Obter modelo
  modelo = RandomForestClassifier(random_state=RANDOM_STATE, max_depth=5)

  # Avaliar modelo
  metricas = avaliar_modelo("Random Forest Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

  return metricas

## **Executar Modelo GradientBoostingClassifier**

In [ ]:
def executar_gradient_boosting_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo GradientBoostingClassifier.
  """
  logger.info("Executando modelo GradientBoostingClassifier")

  # Obter modelo
  modelo = GradientBoostingClassifier(random_state=RANDOM_STATE, max_depth=5)

  # Avaliar modelo
  metricas = avaliar_modelo("Gradient Boosting Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

  return metricas

## **Construir Modelo MLP (PyTorch)**

In [ ]:
def criar_modelo(input_dim=30):
    """
    Cria e retorna o modelo PyTorch Sequential.
    """
    logger.info(f"Criando modelo MLP Sequencial com input_dim={input_dim}")

    model = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

    return model

## **Preparar Dataloaders para uso do MLP**

In [ ]:
def preparar_dataloaders(X_train, y_train, X_val, y_val, batch_size=64):
    """
    Converte os arrays NumPy em Tensores PyTorch e instancia os DataLoaders.
    """
    logger.info("Preparando DataLoaders do PyTorch")

    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train).view(-1, 1)
    X_val_tensor = torch.FloatTensor(X_val)
    y_val_tensor = torch.FloatTensor(y_val.values if hasattr(y_val, 'values') else y_val).view(-1, 1)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

## **Treinar Modelo MLP**

In [ ]:
def treinar_modelo_mlp(model, train_loader, val_loader, epochs=100, lr=0.001, patience=5, min_delta=0.001):
    """
    Loop de treinamento do PyTorch com lógica de Early Stopping integrada.
    """
    logger.info("Iniciando loop de treinamento...")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_loss = None
    patience_counter = 0
    best_model_state = None

    for epoch in range(epochs):
        # --- Fase de Treinamento ---
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # --- Fase de Validação ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                predictions = model(X_batch)
                loss = criterion(predictions, y_batch)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        logger.info(f"Epoch {epoch + 1:03d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # Lógica de Early Stopping
        if best_loss is None:
            best_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif avg_val_loss > best_loss - min_delta:
            patience_counter += 1
            if patience_counter >= patience:
                logger.info(f"Early stopping ativado na época {epoch + 1}! O erro de validação parou de cair.")
                break
        else:
            best_loss = avg_val_loss
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restaura os melhores pesos alcançados durante o treinamento
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        logger.info("Melhor estado do modelo restaurado.")

    return model

## **Obter as Previsões do Modelo MLP**

In [ ]:
def obter_previsoes_mlp(model, X_data):
    """
    Executa a inferência e retorna classes preditas (0 ou 1) e probabilidades brutas.
    """
    logger.info("Obtendo previsões do modelo MLP")

    model.eval()

    X_tensor = torch.FloatTensor(X_data)

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= 0.5).astype(int)

    return preds.flatten(), probs.flatten()

## **Executar Modelo MLP**

In [ ]:
def executar_modelo_mlp(X_train, X_test, y_train, y_test, transformer, balanceador):
    """
    Prepara os dados, executa treinamento e validação, avalia e salva resultados da MLP.
    """
    logger.info("Executando Modelo MLP")

    logger.debug("Aplicando transformação dos dados")
    X_train_trans = transformer.fit_transform(X_train)
    X_test_trans = transformer.transform(X_test)

    logger.debug("Convertendo para Dense Array se for matriz esparsa")
    if hasattr(X_train_trans, "toarray"):
        X_train_trans = X_train_trans.toarray()
    if hasattr(X_test_trans, "toarray"):
        X_test_trans = X_test_trans.toarray()

    # Dividir o conjunto de treino original transformado em sub-treino e validação
    # para monitoramento do early stopping, isolando totalmente o conjunto de teste.
    X_sub_train, X_val, y_sub_train, y_val = train_test_split(
        X_train_trans, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )

    logger.debug("Balanceando os dados")
    X_sub_train_res, y_sub_train_res = balanceador.fit_resample(X_sub_train, y_sub_train)

    train_loader, val_loader = preparar_dataloaders(X_sub_train_res, y_sub_train_res, X_val, y_val, batch_size=64)

    model = criar_modelo(input_dim=X_sub_train_res.shape[1])

    treinar_modelo_mlp(model, train_loader, val_loader, epochs=100, lr=0.001, patience=5)

    preds_train, _ = obter_previsoes_mlp(model, X_train_trans)
    preds_test, _ = obter_previsoes_mlp(model, X_test_trans)

    previsoes = {
        "previsoes_train": preds_train,
        "previsoes_test": preds_test
    }

    nome_execucao = "MLP (PyTorch)"

    metricas = calcular_metricas_treino_teste(nome_execucao, y_train, y_test, previsoes)

    registrar_execucao_mlflow(nome_execucao, metricas, model, True)

    return_data = {
        "modelo": nome_execucao,
        "metricas": metricas
    }

    return return_data

## **Exibir gráfico comparativo métricas/modelo**

In [ ]:
def exibir_grafico_comparativo_metricas_modelo(df_metricas):
  """
  Exibe o gráfico comparativo métricas/modelo.
  """
  logger.info("Exibindo gráfico comparativo métricas/modelo")

  metricas = [
      "Accuracy",
      "Precision",
      "Recall",
      "F1-Score",
      "Overfitting"
  ]

  plt.figure(figsize=(18, 10))

  for i, col in enumerate(metricas):
      plt.subplot(2, 3, i+1)
      sns.barplot(
          data=df_metricas,
          x="Modelo",
          hue="Modelo",
          y=col,
          palette='Set2',
          legend=False
      )
      plt.title(f'Métrica: {col}', fontweight='bold', fontsize=12)
      plt.xlabel('Modelos', fontsize=10)
      plt.ylabel('Valor', fontsize=10)
      plt.xticks(rotation=75) # Rotaciona o nome dos modelos se forem grandes
      plt.grid(axis='y', linestyle='--', alpha=0.7) # Linhas de grade para facilitar leitura

  plt.suptitle('Comparativo de Métricas entre os Modelos', fontsize=16, fontweight='bold', y=0.98)
  plt.tight_layout()
  plt.show()

## **Exibir Comparativos entre Modelos**

In [ ]:
def exibir_comparativos_modelos(metricas_todos_modelos):
  """
  Exibe os comparativos entre os modelos.
  """
  logger.info("Exibindo comparativos entre os modelos")
  df_metricas = pd.DataFrame(metricas_todos_modelos)

  display(df_metricas.head(10))

  exibir_grafico_comparativo_metricas_modelo(df_metricas)

## **Identificar Path do Arquivo de Dados**

In [ ]:
def identificar_path_arquivo_dados():
  """
  Identifica o path do arquivo de dados.
  """
  logger.info("Identificando path do arquivo de dados")

  # Identificar arquivo com fallback de ambientes (Local vs Colab)
  caminho_local = "../data/raw/Telco_customer_churn.xlsx"
  caminho_colab = "/content/Telco_customer_churn.xlsx"

  if os.path.exists(caminho_local):
      path = caminho_local
  elif os.path.exists(caminho_colab):
      path = caminho_colab
  else:
      logger.error("Erro: Base de dados Telco_customer_churn.xlsx não foi encontrada localmente nem no caminho padrão do Colab.")
      sys.exit(1)

  return path

## **Execução Geral do 'Programa'**

In [ ]:
def main():
  logger.info("Iniciando o programa")

  # Configurar o Logger
  configurar_logging(logging.DEBUG)

  # Identificar path do arquivo de dados
  path = identificar_path_arquivo_dados()

  # Carregar dados
  df = carregar_dados(path)

  # Tratamento dos dados
  df = tratar_dados(df)

  # Separar dados de treino e teste
  X_train, X_test, y_train, y_test = separar_dados_treino_teste(df)

  # Obter transformer
  transformer = construir_transformer_hot_encoding(X_train)

  # Obter balanceador
  balanceador = construir_balanceador()

  # Configurar o MLFlow
  configurar_mlflow()

  # Buffer para coletar as métricas de todos os modelos
  metricas_todos_modelos = []

  # Executar o modelo LogisticRegression
  metricas = executar_logistic_regression(X_train, X_test, y_train, y_test, transformer, balanceador)
  registrar_metricas_modelo(metricas["modelo"], metricas["metricas"], metricas_todos_modelos)

  # Executar o modelo DecisionTreeClassifier
  metricas = executar_decision_tree_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)
  registrar_metricas_modelo(metricas["modelo"], metricas["metricas"], metricas_todos_modelos)

  # Executar o modelo RandomForestClassifier
  metricas = executar_random_forest_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)
  registrar_metricas_modelo(metricas["modelo"], metricas["metricas"], metricas_todos_modelos)

  # Executar o modelo GradientBoostingClassifier
  metricas = executar_gradient_boosting_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)
  registrar_metricas_modelo(metricas["modelo"], metricas["metricas"], metricas_todos_modelos)

  # Executar o modelo MLP (PyTorch)
  metricas = executar_modelo_mlp(X_train, X_test, y_train, y_test, transformer, balanceador)
  registrar_metricas_modelo(metricas["modelo"], metricas["metricas"], metricas_todos_modelos)

  # Exibir comparativo
  exibir_comparativos_modelos(metricas_todos_modelos)

  logger.info("Programa finalizado")

In [ ]:
if __name__ == "__main__":
    main()